In [ ]:
import pandas as pd
import os
import time
import pyarrow as pa
import pyarrow.parquet as pq

print('Library berhasil diimport')
print(f'Pandas version  : {pd.__version__}')
print(f'PyArrow version : {pa.__version__}')

Library berhasil diimport
Pandas version  : 2.2.2
PyArrow version : 18.1.0


In [ ]:
import google.colab
print("Google Colab terdeteksi")

Google Colab terdeteksi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.listdir('/content/drive/MyDrive/ABD')

['data', 'bronze', 'figures']

In [ ]:
DATA_DIR   = '/content/drive/MyDrive/ABD/data/raw'
BRONZE_DIR = '/content/drive/MyDrive/ABD/data/bronze'

In [ ]:
#CEK DATASET
import pandas as pd

df = pd.read_excel('/content/drive/MyDrive/ABD/data/raw/ghgp_data_2014.xlsx')
df.head()

,Summary data collected by the Greenhouse Gas Reporting Program for 2014,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 56,Unnamed: 57,Unnamed: 58,Unnamed: 59,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65
0,This data was reported to EPA by facilities as...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,All emissions data is presented in units of me...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Facility Id,FRS Id,Facility Name,City,State,Zip Code,Address,County,Latitude,Longitude,...,Titanium Dioxide Production,Underground Coal Mines,Zinc Production,Municipal Landfills,Industrial Wastewater Treatment,Manufacture of Electric Transmission and Distr...,Industrial Waste Landfills,Is some CO2 collected on-site and used to manu...,Is some CO2 reported as emissions from the aff...,Does the facility employ continuous emissions ...
3,1004377,110043803578,121 REGIONAL DISPOSAL FACILITY,MELISSA,TX,75454,3820 SAM RAYBURN HIGHWAY,COLLIN COUNTY,33.29857,-96.53586,...,NaN,NaN,NaN,241883.5,NaN,NaN,NaN,N,N,N
4,1010040,110070082056,15-18565/15-18662,Hazard,KY,40701,4200 S. Hwy 15,PERRY COUNTY,37.219099,-83.156046,...,NaN,225708.25,NaN,NaN,NaN,NaN,NaN,N,N,N


In [ ]:
# Rentang tahun yang digunakan
YEARS = list(range(2014, 2024))  # [2014, 2015, ..., 2023]

# Buat folder output kalau belum ada
os.makedirs(BRONZE_DIR, exist_ok=True)

print(f'Data dir    : {DATA_DIR}')
print(f'Bronze dir  : {BRONZE_DIR}')
print(f'Tahun       : {YEARS[0]}–{YEARS[-1]} ({len(YEARS)} tahun)')

Data dir    : /content/drive/MyDrive/ABD/data/raw
Bronze dir  : /content/drive/MyDrive/ABD/data/bronze
Tahun       : 2014–2023 (10 tahun)


In [ ]:
# Kolom yang akan diambil dari setiap file
KEEP_COLS = [
    'Facility Id',
    'Facility Name',
    'City',
    'State',
    'Primary NAICS Code',
    'Industry Type (sectors)',
    'Total reported direct emissions',
    'CO2 emissions (non-biogenic) ',    # spasi di akhir memang ada di file asli EPA
    'Methane (CH4) emissions ',          # spasi di akhir memang ada di file asli EPA
    'Nitrous Oxide (N2O) emissions ',    # spasi di akhir memang ada di file asli EPA
]

def get_sheet_name(year):
    """
    EPA mengubah nama sheet mulai tahun 2018.
    Fungsi ini mengembalikan nama sheet yang benar untuk setiap tahun.
    """
    if year >= 2018:
        return 'Direct Point Emitters'
    else:
        return 'Direct Emitters'

print('Konfigurasi kolom siap')
print(f'Jumlah kolom yang diambil: {len(KEEP_COLS)}')

Konfigurasi kolom siap
Jumlah kolom yang diambil: 10


In [ ]:
start_time = time.time()  # Catat waktu mulai untuk perbandingan dengan Spark nanti

frames = []  # List kosong untuk menampung DataFrame tiap tahun

for year in YEARS:
    path  = os.path.join(DATA_DIR, f'ghgp_data_{year}.xlsx')
    sheet = get_sheet_name(year)

    # Baca file Excel
    # header=3 → nama kolom ada di baris ke-4 (index 3)
    df = pd.read_excel(path, sheet_name=sheet, header=3)

    # Ambil hanya kolom yang kita butuhkan
    # (gunakan list comprehension untuk filter kolom yang benar-benar ada)
    available_cols = [c for c in KEEP_COLS if c in df.columns]
    df = df[available_cols].copy()

    # Tambahkan kolom tahun
    df['year'] = year

    frames.append(df)
    print(f'  {year}: {len(df):,} fasilitas dibaca')

# Gabungkan semua tahun menjadi satu DataFrame
# ignore_index=True → reset index supaya tidak ada index duplikat
df_bronze = pd.concat(frames, ignore_index=True)

ingestion_time = time.time() - start_time

print(f'\n✓ Ingestion selesai dalam {ingestion_time:.1f} detik')
print(f'  Total baris  : {df_bronze.shape[0]:,}')
print(f'  Total kolom  : {df_bronze.shape[1]}')

  2014: 7,349 fasilitas dibaca
  2015: 7,242 fasilitas dibaca
  2016: 6,572 fasilitas dibaca
  2017: 6,488 fasilitas dibaca
  2018: 6,581 fasilitas dibaca
  2019: 6,580 fasilitas dibaca
  2020: 6,556 fasilitas dibaca
  2021: 6,529 fasilitas dibaca
  2022: 6,514 fasilitas dibaca
  2023: 6,470 fasilitas dibaca

✓ Ingestion selesai dalam 45.6 detik
  Total baris  : 66,881
  Total kolom  : 11


In [ ]:
print('=== VALIDASI BRONZE LAYER ===')

# 1. Jumlah baris per tahun
print('\n[1] Jumlah fasilitas per tahun:')
print(df_bronze.groupby('year').size().to_string())

# 2. Missing values
print('\n[2] Missing values per kolom:')
missing = df_bronze.isnull().sum()
missing_pct = (df_bronze.isnull().mean() * 100).round(1)
for col in df_bronze.columns:
    print(f'  {col[:45]:<45}: {missing[col]:>6,} ({missing_pct[col]:.1f}%)')

# 3. Tipe data
print('\n[3] Tipe data per kolom:')
print(df_bronze.dtypes.to_string())

# 4. Sample data
print('\n[4] Contoh 5 baris pertama:')
display_cols = ['Facility Id', 'Facility Name', 'State',
                'Industry Type (sectors)', 'Total reported direct emissions', 'year']
print(df_bronze[display_cols].head(5).to_string())

=== VALIDASI BRONZE LAYER ===

[1] Jumlah fasilitas per tahun:
year
2014    7349
2015    7242
2016    6572
2017    6488
2018    6581
2019    6580
2020    6556
2021    6529
2022    6514
2023    6470

[2] Missing values per kolom:
  Facility Id                                  :      0 (0.0%)
  Facility Name                                :      0 (0.0%)
  City                                         :      0 (0.0%)
  State                                        :      0 (0.0%)
  Primary NAICS Code                           :      9 (0.0%)
  Industry Type (sectors)                      :      0 (0.0%)
  Total reported direct emissions              :      0 (0.0%)
  CO2 emissions (non-biogenic)                 :  7,034 (10.5%)
  Methane (CH4) emissions                      :  1,150 (1.7%)
  Nitrous Oxide (N2O) emissions                : 11,015 (16.5%)
  year                                         :      0 (0.0%)

[3] Tipe data per kolom:
Facility Id                          int64
Facilit

In [ ]:
emisi_col = 'Total reported direct emissions'

print('=== STATISTIK DESKRIPTIF EMISI ===')
stats = df_bronze[emisi_col].describe()
print(f'  Count  : {stats["count"]:>15,.0f} baris')
print(f'  Mean   : {stats["mean"]:>15,.2f} metrik ton CO2e')
print(f'  Median : {df_bronze[emisi_col].median():>15,.2f} metrik ton CO2e')
print(f'  Std    : {stats["std"]:>15,.2f}')
print(f'  Min    : {stats["min"]:>15,.2f} metrik ton CO2e')
print(f'  Max    : {stats["max"]:>15,.2f} metrik ton CO2e')

ratio = stats['mean'] / df_bronze[emisi_col].median()
print(f'\n  Rasio mean/median : {ratio:.1f}x')
if ratio > 3:
    print('  → Distribusi sangat skewed! Log transformation diperlukan di Silver Layer.')
else:
    print('  → Distribusi relatif normal.')

# Top 5 sektor berdasarkan total emisi
print('\n=== TOP 5 SEKTOR BERDASARKAN TOTAL EMISI ===')
top_sectors = (
    df_bronze.groupby('Industry Type (sectors)')[emisi_col]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
for sector, total in top_sectors.items():
    print(f'  {sector[:50]:<50}: {total:>20,.0f} ton CO2e')

=== STATISTIK DESKRIPTIF EMISI ===
  Count  :          66,881 baris
  Mean   :      400,450.95 metrik ton CO2e
  Median :       63,965.78 metrik ton CO2e
  Std    :    1,225,084.15
  Min    :       -1,196.25 metrik ton CO2e
  Max    :   21,775,439.59 metrik ton CO2e

  Rasio mean/median : 6.3x
  → Distribusi sangat skewed! Log transformation diperlukan di Silver Layer.

=== TOP 5 SEKTOR BERDASARKAN TOTAL EMISI ===
  Power Plants                                      :       16,490,917,635 ton CO2e
  Minerals                                          :        1,109,289,401 ton CO2e
  Chemicals                                         :        1,035,087,089 ton CO2e
  Other                                             :        1,022,575,394 ton CO2e
  Waste                                             :          967,412,755 ton CO2e


In [ ]:
# Path output
parquet_path = os.path.join(BRONZE_DIR, 'ghgrp_bronze.parquet')
csv_path     = os.path.join(BRONZE_DIR, 'ghgrp_bronze.csv')

# Simpan ke Parquet
df_bronze.to_parquet(parquet_path, index=False, compression='snappy')
# compression='snappy' → algoritma kompresi cepat yang umum dipakai dengan Spark

# Simpan juga ke CSV untuk perbandingan ukuran
df_bronze.to_csv(csv_path, index=False)

# Hitung ukuran file
size_parquet = os.path.getsize(parquet_path) / (1024 * 1024)  # dalam MB
size_csv     = os.path.getsize(csv_path)     / (1024 * 1024)
compression  = (1 - size_parquet / size_csv) * 100

print('=== HASIL PENYIMPANAN BRONZE LAYER ===')
print(f'  Ukuran CSV     : {size_csv:.2f} MB')
print(f'  Ukuran Parquet : {size_parquet:.2f} MB')
print(f'  Kompresi       : {compression:.1f}% lebih kecil dari CSV')
print(f'\n  File disimpan di:')
print(f'    {parquet_path}')
print(f'    {csv_path}')
print(f'\n✓ Bronze Layer selesai!')
print(f'  Total waktu ingestion: {ingestion_time:.1f} detik')
print(f'  Catatan waktu ini untuk dibandingkan dengan Spark di Silver Layer.')

=== HASIL PENYIMPANAN BRONZE LAYER ===
  Ukuran CSV     : 6.96 MB
  Ukuran Parquet : 2.10 MB
  Kompresi       : 69.9% lebih kecil dari CSV

  File disimpan di:
    /content/drive/MyDrive/ABD/data/bronze/ghgrp_bronze.parquet
    /content/drive/MyDrive/ABD/data/bronze/ghgrp_bronze.csv

✓ Bronze Layer selesai!
  Total waktu ingestion: 45.6 detik
  Catatan waktu ini untuk dibandingkan dengan Spark di Silver Layer.


In [ ]:
print('=' * 55)
print('       RINGKASAN BRONZE LAYER — SIAP DILAPORKAN')
print('=' * 55)
print(f'  Dataset        : EPA GHGRP')
print(f'  Rentang tahun  : 2014–2023 (10 tahun)')
print(f'  Total fasilitas: {df_bronze["Facility Id"].nunique():,} unik')
print(f'  Total baris    : {df_bronze.shape[0]:,}')
print(f'  Total kolom    : {df_bronze.shape[1]}')
print(f'  Sektor industri: {df_bronze["Industry Type (sectors)"].nunique()} unik')
print(f'  Negara bagian  : {df_bronze["State"].nunique()} state')
print(f'  Ukuran Parquet : {size_parquet:.2f} MB')
print(f'  Ukuran CSV     : {size_csv:.2f} MB')
print(f'  Kompresi       : {compression:.1f}%')
print(f'  Waktu ingestion: {ingestion_time:.1f} detik (Pandas baseline)')
print()
print('  Catatan untuk Silver Layer:')
print('  - CO2 emissions: ~10.5% missing → perlu strategi imputasi/drop')
print('  - N2O emissions: ~16.5% missing → perlu strategi imputasi/drop')
print('  - Distribusi emisi sangat skewed → log transformation diperlukan')
print('=' * 55)
print()
print('Langkah selanjutnya: buka Silver_Layer.ipynb')

       RINGKASAN BRONZE LAYER — SIAP DILAPORKAN
  Dataset        : EPA GHGRP
  Rentang tahun  : 2014–2023 (10 tahun)
  Total fasilitas: 8,462 unik
  Total baris    : 66,881
  Total kolom    : 11
  Sektor industri: 70 unik
  Negara bagian  : 54 state
  Ukuran Parquet : 2.10 MB
  Ukuran CSV     : 6.96 MB
  Kompresi       : 69.9%
  Waktu ingestion: 45.6 detik (Pandas baseline)

  Catatan untuk Silver Layer:
  - CO2 emissions: ~10.5% missing → perlu strategi imputasi/drop
  - N2O emissions: ~16.5% missing → perlu strategi imputasi/drop
  - Distribusi emisi sangat skewed → log transformation diperlukan

Langkah selanjutnya: buka Silver_Layer.ipynb
